# Session 8: Replication and Sharding

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in []:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-08_replica_set_sim.py`

In [2]:
# Session 8 · A replica set in miniature: oplog, write concern and failover
# This is a teaching model (no real servers): it mimics how a 3-member replica set behaves.

class Member:
    def __init__(self, name, priority=1):
        self.name, self.priority, self.up = name, priority, True
        self.data, self.oplog = {}, []          # oplog = ordered list of operations applied

    def apply(self, op):
        self.oplog.append(op)
        kind, key, val = op
        if kind == "set": self.data[key] = val
        elif kind == "del": self.data.pop(key, None)

class ReplicaSet:
    def __init__(self, members):
        self.members = members
        self.primary = max(members, key=lambda m: m.priority)
        print(f"rs.initiate(): {self.primary.name} elected PRIMARY")

    def write(self, key, val, w=1):
        if not self.primary or not self.primary.up:
            raise RuntimeError("NotWritablePrimary: no primary available")
        op = ("set", key, val)
        self.primary.apply(op)
        acks = 1
        for m in self.members:                   # secondaries tail the primary's oplog
            if m is not self.primary and m.up:
                m.apply(op); acks += 1
        needed = len(self.members) // 2 + 1 if w == "majority" else w
        status = "acknowledged" if acks >= needed else "WriteConcernError (waiting for replication)"
        print(f"  write {key}={val!r:<10} w={w!s:<8} acks={acks}/{len(self.members)} -> {status}")

    def elect(self):
        alive = [m for m in self.members if m.up]
        if len(alive) <= len(self.members) // 2:
            self.primary = None
            print(f"  election FAILED: only {len(alive)} of {len(self.members)} members up (no majority) -> read-only")
            return
        # the most up-to-date member with the highest priority wins
        self.primary = max(alive, key=lambda m: (len(m.oplog), m.priority))
        print(f"  election: {self.primary.name} is the new PRIMARY")

    def status(self):
        for m in self.members:
            role = "PRIMARY" if m is self.primary else ("SECONDARY" if m.up else "DOWN")
            print(f"    {m.name:<6} {role:<9} oplog={len(m.oplog)} data={m.data}")

rs = ReplicaSet([Member("node1", priority=2), Member("node2"), Member("node3")])
rs.write("stock:101", 40)
rs.write("stock:102", 15, w="majority")
rs.status()

print("\nnode1 crashes")
rs.members[0].up = False
rs.elect()
rs.write("stock:101", 39, w="majority")
rs.status()

print("\nnode3 also goes down")
rs.members[2].up = False
rs.write("stock:103", 6, w=1)
rs.write("stock:104", 0, w="majority")
rs.elect()
try:
    rs.write("stock:105", 9)
except RuntimeError as e:
    print("  ", e)

rs.initiate(): node1 elected PRIMARY
  write stock:101=40         w=1        acks=3/3 -> acknowledged
  write stock:102=15         w=majority acks=3/3 -> acknowledged
    node1  PRIMARY   oplog=2 data={'stock:101': 40, 'stock:102': 15}
    node2  SECONDARY oplog=2 data={'stock:101': 40, 'stock:102': 15}
    node3  SECONDARY oplog=2 data={'stock:101': 40, 'stock:102': 15}

node1 crashes
  election: node2 is the new PRIMARY
  write stock:101=39         w=majority acks=2/3 -> acknowledged
    node1  DOWN      oplog=2 data={'stock:101': 40, 'stock:102': 15}
    node2  PRIMARY   oplog=3 data={'stock:101': 39, 'stock:102': 15}
    node3  SECONDARY oplog=3 data={'stock:101': 39, 'stock:102': 15}

node3 also goes down
  write stock:103=6          w=1        acks=1/3 -> acknowledged
  write stock:104=0          w=majority acks=1/3 -> WriteConcernError (waiting for replication)
  election FAILED: only 1 of 3 members up (no majority) -> read-only
   NotWritablePrimary: no primary available


## Example: `session-08_sharding_sim.py`

In [3]:
# Session 8 · How the shard key decides where documents live
# Teaching model of 3 shards, comparing a ranged key on a growing value with a hashed key.
import hashlib
from collections import Counter

SHARDS = ["shardA", "shardB", "shardC"]
order_ids = list(range(1, 30_001))              # monotonically increasing _id values
new_ids = list(range(30_001, 33_001))           # the next 3,000 inserts

# Ranged sharding: chunks own contiguous ranges of the key
ranges = [(1, 10_000, "shardA"), (10_001, 20_000, "shardB"), (20_001, float("inf"), "shardC")]
def ranged(k):
    for lo, hi, s in ranges:
        if lo <= k <= hi: return s

def hashed(k):
    h = int(hashlib.md5(str(k).encode()).hexdigest(), 16)
    return SHARDS[h % len(SHARDS)]

def report(title, fn, keys):
    c = Counter(fn(k) for k in keys)
    print(f"  {title:<34}" + "  ".join(f"{s}={c.get(s, 0):>6,}" for s in SHARDS))

print("Existing 30,000 orders")
report("ranged on _id", ranged, order_ids)
report("hashed on _id", hashed, order_ids)
print("\nNext 3,000 inserts (where does the write load go?)")
report("ranged on _id  (HOT SHARD!)", ranged, new_ids)
report("hashed on _id  (spread out)", hashed, new_ids)

print("\nQuery routing by mongos")
print("  find({_id: 15000})            ranged ->", ranged(15000), "(targeted, 1 shard)")
print("  find({_id: {$gte: 100, $lte: 200}}) ranged ->", sorted({ranged(k) for k in range(100, 201)}), "(targeted)")
print("  same range query, hashed      ->", sorted({hashed(k) for k in range(100, 201)}), "(scatter-gather)")
print("  find({city: 'Lahore'})        -> all shards (field is not the shard key)")

# A low-cardinality key cannot be split further: jumbo chunks
cities = ["Karachi"] * 18_000 + ["Lahore"] * 8_000 + ["Islamabad"] * 4_000
c = Counter(cities)
print("\nShard key {city: 1}: only", len(c), "distinct values ->", dict(c))
print("  The 'Karachi' chunk can never be split, so one shard holds 60% of the data (a jumbo chunk).")

Existing 30,000 orders
  ranged on _id                     shardA=10,000  shardB=10,000  shardC=10,000
  hashed on _id                     shardA= 9,926  shardB=10,101  shardC= 9,973

Next 3,000 inserts (where does the write load go?)
  ranged on _id  (HOT SHARD!)       shardA=     0  shardB=     0  shardC= 3,000
  hashed on _id  (spread out)       shardA=   986  shardB= 1,028  shardC=   986

Query routing by mongos
  find({_id: 15000})            ranged -> shardB (targeted, 1 shard)
  find({_id: {$gte: 100, $lte: 200}}) ranged -> ['shardA'] (targeted)
  same range query, hashed      -> ['shardA', 'shardB', 'shardC'] (scatter-gather)
  find({city: 'Lahore'})        -> all shards (field is not the shard key)

Shard key {city: 1}: only 3 distinct values -> {'Karachi': 18000, 'Lahore': 8000, 'Islamabad': 4000}
  The 'Karachi' chunk can never be split, so one shard holds 60% of the data (a jumbo chunk).
